In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from uncertainties import ufloat
from uncertainties import unumpy as unp
from functions import conversion

In [2]:
h1 = unp.uarray([-200,72], [5,5]) # coordinates in virt m 
h2 = unp.uarray([88,-74 ], [5,5])# virt m

print(conversion(h1,h2))

0.1428+/-0.0031


In [3]:
def theta(t, A, tau, w, phase, h):
        return A * np.exp(-t/tau) * np.sin(w*t + phase) + h

def lin(x,a,b):
        return a*x + b

def rotate_data(t, xdata, ydata):
    popt, pcov = curve_fit(lin, xdata=xdata, ydata=ydata)
    phi = np.arctan(popt[0])
    
    x = xdata * np.cos(-phi) - (ydata - popt[1]) * np.sin(-phi)
    y = xdata * np.sin(-phi) + (ydata - popt[1]) * np.cos(-phi)

    return x, y

from functions_legacy import conversion, rotate_data

def theta(t, A, tau, w, phase, h):
        return A * np.exp(-t/tau) * np.sin(w*t + phase) + h

def lin(x,a,b):
        return a*x + b

def fit( t, xdata, ydata,name, p0: list = [180, 2000, 0.017, -1.5, 630], xlabel=False, cutoff: int = 0,cutoffend:  int = -1, zero_pos: tuple[float | int] | None = None,
        h1: list[ufloat] | None = None, h2: list[ufloat] | None = None, plot: bool = False):

    x, y, x0 = rotate_data(xdata, ydata, zero_pos=zero_pos)
    
    popt, pcov = curve_fit(theta, t[cutoff:cutoffend], x[cutoff:cutoffend], p0=p0) 

    if x0 != None:
        o = ufloat(popt[4], pcov[4,4]**0.5) - x0
        result = np.append(unp.uarray(popt, np.sqrt(np.diag(pcov))), o)
    else:
        result = unp.uarray(popt, np.sqrt(np.diag(pcov)))

    if plot:
        print('haha still no plot. DA SCHAUSCH HER')
    return result


In [4]:
def tag_it(x: ufloat, tag: str):
    return ufloat(x.n, x.s, tag=tag)

In [5]:
# calculate G , incl uncertainties
def G_12(R, theta1, theta2, T, conversion_factor, l, M):
    #technical drawing
    m = 0.028  #0.028 #kg tech drawing

    #I = m*l**2/2 #kg m^2, using MIT estimated formula
    
    #l = tag_it(unp.uarray(0.12, 0.001)-unp.uarray(0.0171, 0.001), tag='l')
    #l=unp.uarray(0.12, 0.001)
    rk = ufloat(0.0171, 0.00001, tag='r_k')# radius of pendulum spheres
    #I = 2*m *( 2/5 * rk**2 + (l/2)**2)  #improvement of I
    I = m*l**2/2 #kg m^2, using MIT estimated formula
    
    # laser meas
    L = ufloat(4.321, 0.001, tag='L') #m
        
    #measured:
    #M = ufloat(1.5, 0.01, tag='M')  #kg +/-10g
    
    #delta_r = 0.0
    #ADJUST r1, r2, r3
    #r = unp.uarray( R  + delta_r, 0.003) #m  51.722+/-0.011 #meas with messschieber in the air, could be improved

        
    #fit: T0, dtheta1, dtheta2
    h1 = theta1 * conversion_factor #conversion factor for vid setup 1 in cm/'m'. used squared addition here
    h2 = theta2 * conversion_factor
    
    #dtheta= unp.arctan( (theta2 - theta1)*0.01 / 4.321) #0.01 go from cm to m
    dtheta= unp.arctan( ((h2 - h1)*0.5)*0.01 / 4.321) #0.01 go from cm to m
        
    #units kg m^2 /s^2  * m^2/kg^2/m
    G = (r)**2 * (l)/(8*M) * (2*np.pi/T)**2 * dtheta  #using MIT I that cancels, we could get a correction here

    #print('eq1:',h1,'eq2:',h2, 'in cm')
    
    return G

In [6]:
l = tag_it(unp.uarray(0.12, 0.001)-unp.uarray(0.0171, 0.001), tag='l')
M = ufloat(1.5, 0.01, tag='M')  #kg +/-10g

In [19]:
#conv = tag_it(conversion(h1,h2), tag='text{conversion}')

T = tag_it(2 * np.pi / fit(*np.loadtxt('m8/m8_p0.txt', skiprows=2, unpack=True),'T', cutoff=500)[2], tag='T')

print('period from zero measurement = ', T)

theta0 = tag_it(fit(*np.loadtxt('m8/m8_p0.txt', skiprows=2, unpack=True), 'M8 zero', cutoff=500, plot=True)[4],
                tag=r'\theta_0')
theta1 = tag_it(fit(*np.loadtxt('m8/m8_p1_r4.txt', skiprows=2, unpack=True), 'M8 position 1', cutoff=500, plot=True)[4],
          tag=r'\theta_1')
theta2 = tag_it(fit(*np.loadtxt('m8/m8_p2_r4.txt', skiprows=2, unpack=True), 'M8 position 2',xlabel=True, cutoff=500, plot=True)[4],
          tag=r'\theta_2')

R4 = 0.068

delta_r = 0.0
r = ufloat(R4 + delta_r, 0.003, tag='r') 

conversion_factor =tag_it(conversion(h1,h2), tag=r'\text{conversion}')
print('G12 =',G_12(r, theta1, theta2, T, conversion_factor, l, M)) # *2 for G01,2 
print('G01 =',G_12(r, theta1, theta0, T, conversion_factor, l, M)*2)
print('G02 =',G_12(r, theta0, theta2, T, conversion_factor, l, M)*2)

period from zero measurement =  378.93+/-0.11
haha still no plot. DA SCHAUSCH HER
haha still no plot. DA SCHAUSCH HER
haha still no plot. DA SCHAUSCH HER
G12 = (6.8+/-0.6)e-11
G01 = (7.3+/-0.7)e-11
G02 = (6.3+/-0.6)e-11


In [23]:
print('G12')
G = G_12(r, theta1, theta2, T, conversion_factor, l, M)
for (var, error) in G.error_components().items():
    print(f"${var.tag}$ & " + f"${var:L}$" + f" & {round((error / G.s)**2, 5)}\\\\")

G12
$\text{conversion}$ & $0.1428 \pm 0.0031$ & 0.05694\\
$\theta_1$ & $-39.735 \pm 0.025$ & 5e-05\\
$\theta_2$ & $-1.957 \pm 0.025$ & 5e-05\\
$T$ & $378.93 \pm 0.11$ & 4e-05\\
$M$ & $1.500 \pm 0.010$ & 0.00523\\
$l$ & $0.1029 \pm 0.0014$ & 0.02221\\
$r$ & $0.0680 \pm 0.0030$ & 0.91548\\
